# ViT MIMIC Target Experiment Runner

Train, evaluate, explain, cluster, and plot the existing time-series ViT workflow for selected MIMIC targets and prediction-window settings.

This notebook is intentionally parameterized instead of duplicated per target. Change `TARGETS`, `WINDOWS`, and the config objects below to run a different experiment set.

## Imports and Logging

In [ ]:
from __future__ import annotations

import json
import logging
import shutil
import sys
from dataclasses import replace
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "interpretable_ts_vit").exists():
    PROJECT_ROOT = Path.cwd().parent.parent
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from interpretable_ts_vit.binning import TimeSeriesBinner
from interpretable_ts_vit.config import ClusterConfig, DataConfig, ExplainConfig, ModelConfig, TrainConfig
from interpretable_ts_vit.data_modules import GenericCSVDataModule
from interpretable_ts_vit.datasets.mimic import configured_variables_for_target, load_mimic_targets_config
from interpretable_ts_vit.model_modules import ViTTimeSeriesModule

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)
logger = logging.getLogger("vit_experiment_runner")
logger.info("Project root: %s", PROJECT_ROOT)

## Experiment Settings

In [ ]:
EXPERIMENT_NAME = "vit_baseline"

TARGETS = [
    "hypoglycemia",
    "hypokalemia",
    "hypotension",
]

WINDOWS = [
    "obs24_target8_gap0",
    "obs24_target8_gap2",
]

SPLIT = "test"
DATA_ROOT = PROJECT_ROOT / "data" / "mimic_targets"
PROCESSED_ROOT = DATA_ROOT / "processed"
RUN_ROOT = PROJECT_ROOT / "runs" / "mimic_targets" / EXPERIMENT_NAME
MIMIC_TARGETS_CONFIG_PATH = PROJECT_ROOT / "configs" / "datasets" / "mimic" / "targets.yaml"
REBUILD_PROCESSED_TENSORS_ON_VARIABLE_MISMATCH = True

data_config = DataConfig(
    granularity="30min",
    aggregation="mean",
    val_fraction=0.2,
    test_fraction=0.2,
    random_state=13,
)

model_config = ModelConfig(
    patch_size=(1, 4),
    embed_dim=64,
    depth=2,
    num_heads=4,
    mlp_ratio=2.0,
    dropout=0.1,
)

train_config = TrainConfig(
    batch_size=16,
    epochs=10,
    learning_rate=1e-3,
    weight_decay=1e-4,
    device="auto",
    early_stopping_patience=3,
    restore_best_model=True,
    verbose=True,
)

explain_config = ExplainConfig(
    method="grad_attention_rollout",
    target_class=None,
    batch_size=16,
)

cluster_config = ClusterConfig(
    feature_mode="autoencoder",
    method="kmeans",
    n_clusters=8,
    autoencoder_latent_dim=16,
    autoencoder_epochs=50,
    autoencoder_learning_rate=1e-3,
    autoencoder_batch_size=32,
    autoencoder_early_stopping_patience=10,
    plot_mode="value_with_importance_opacity",
    importance_threshold=None,
    show_values=True,
    use_normal_ranges=False,
)

RUN_ROOT

## Helpers

In [ ]:
def experiment_paths(target: str, window: str) -> dict[str, Path]:
    source_dir = DATA_ROOT / window / target
    processed_dir = PROCESSED_ROOT / window / target
    run_dir = RUN_ROOT / window / target
    return {
        "source_dir": source_dir,
        "records": source_dir / "records.csv",
        "labels": source_dir / "labels.csv",
        "processed_dir": processed_dir,
        "run_dir": run_dir,
        "metrics": run_dir / f"{SPLIT}_evaluation_metrics.json",
        "predictions": run_dir / f"{SPLIT}_predictions.csv",
        "explanations": run_dir / "explanations" / SPLIT,
        "clusters": run_dir / "clusters" / SPLIT,
        "cluster_assignments": run_dir / "clusters" / SPLIT / "cluster_assignments.csv",
        "cluster_heatmaps": run_dir / "cluster_heatmaps" / SPLIT,
        "cluster_centroid_heatmaps": run_dir / "cluster_centroid_heatmaps" / SPLIT,
    }


def allowed_variables_for_target(target: str) -> list[str]:
    mimic_config = load_mimic_targets_config(MIMIC_TARGETS_CONFIG_PATH)
    allowed_variables = configured_variables_for_target(mimic_config, target)
    if not allowed_variables:
        raise ValueError(f"No YAML-configured variables found for target={target!r} in {MIMIC_TARGETS_CONFIG_PATH}")
    logger.info("Loaded %d YAML-configured variables for target=%s", len(allowed_variables), target)
    return allowed_variables


def data_config_for_target(target: str) -> DataConfig:
    return replace(data_config, allowed_variables=allowed_variables_for_target(target))


def ensure_processed_cache_matches_yaml(data: GenericCSVDataModule, allowed_variables: list[str], target: str, window: str) -> None:
    if not data.binner_path.exists():
        return
    binner = TimeSeriesBinner.load(data.binner_path)
    extra_variables = sorted(set(binner.variable_vocab_) - set(allowed_variables))
    if not extra_variables:
        logger.info("Prepared tensor cache matches YAML variable allow-list for target=%s window=%s", target, window)
        return
    message = (
        f"Prepared tensor cache at {data.processed_dir} contains {len(extra_variables)} variable(s) "
        f"not configured for target={target!r}: {extra_variables}"
    )
    if not REBUILD_PROCESSED_TENSORS_ON_VARIABLE_MISMATCH:
        raise ValueError(message)
    processed_dir = Path(data.processed_dir).resolve()
    processed_root = PROCESSED_ROOT.resolve()
    if processed_root not in processed_dir.parents and processed_dir != processed_root:
        raise ValueError(f"Refusing to rebuild processed tensors outside {processed_root}: {processed_dir}")
    logger.warning("%s. Rebuilding processed tensors from YAML-filtered records.", message)
    shutil.rmtree(processed_dir)


def build_data_module(target: str, window: str) -> GenericCSVDataModule:
    paths = experiment_paths(target, window)
    missing = [name for name in ("records", "labels") if not paths[name].exists()]
    if missing:
        missing_paths = ", ".join(str(paths[name]) for name in missing)
        raise FileNotFoundError(f"Missing input file(s) for {window}/{target}: {missing_paths}")
    target_data_config = data_config_for_target(target)
    data = GenericCSVDataModule(
        records_path=paths["records"],
        labels_path=paths["labels"],
        processed_dir=paths["processed_dir"],
        data_config=target_data_config,
    )
    ensure_processed_cache_matches_yaml(data, target_data_config.allowed_variables or [], target, window)
    return data


def build_model_module(target: str, window: str) -> ViTTimeSeriesModule:
    paths = experiment_paths(target, window)
    return ViTTimeSeriesModule(
        run_dir=paths["run_dir"],
        model_config=model_config,
        train_config=train_config,
        explain_config=explain_config,
        cluster_config=cluster_config,
    )


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as fh:
        return json.load(fh)


def flatten_metrics(metrics: dict, prefix: str = "") -> dict:
    flat = {}
    for key, value in metrics.items():
        name = f"{prefix}{key}" if not prefix else f"{prefix}.{key}"
        if isinstance(value, dict):
            flat.update(flatten_metrics(value, name))
        else:
            flat[name] = value
    return flat

## Single Run

In [ ]:
def run_vit_experiment(target: str, window: str, split: str = SPLIT, display_heatmaps: bool = True) -> dict:
    paths = experiment_paths(target, window)
    logger.info("Starting experiment target=%s window=%s", target, window)
    logger.info("Records: %s", paths["records"])
    logger.info("Processed tensors: %s", paths["processed_dir"])
    logger.info("Run output: %s", paths["run_dir"])

    data = build_data_module(target, window)
    model = build_model_module(target, window)

    logger.info("Stage prepare-data started for target=%s window=%s", target, window)
    data.prepare()
    logger.info("Stage prepare-data finished for target=%s window=%s", target, window)

    logger.info("Stage train started for target=%s window=%s", target, window)
    train_metrics = model.fit(data)
    logger.info("Stage train finished for target=%s window=%s", target, window)

    logger.info("Stage evaluate started for target=%s window=%s split=%s", target, window, split)
    evaluation_metrics = model.evaluate(data, split=split)
    logger.info("Stage evaluate finished for target=%s window=%s split=%s", target, window, split)

    logger.info("Stage explain started for target=%s window=%s split=%s", target, window, split)
    explanations_dir = model.explain(data, split=split)
    logger.info("Stage explain finished for target=%s window=%s split=%s", target, window, split)

    logger.info("Stage cluster started for target=%s window=%s split=%s", target, window, split)
    clusters_dir = model.cluster_explanations(data, split=split)
    logger.info("Stage cluster finished for target=%s window=%s split=%s", target, window, split)

    logger.info("Stage plot started for target=%s window=%s split=%s", target, window, split)
    heatmaps_dir = model.plot_cluster_values(data, split=split)
    logger.info("Stage plot finished for target=%s window=%s split=%s", target, window, split)

    if display_heatmaps:
        model.display_cluster_heatmaps(split=split)

    logger.info("Finished experiment target=%s window=%s", target, window)
    return {
        "target": target,
        "window": window,
        "split": split,
        "train_metrics": train_metrics,
        "evaluation_metrics": evaluation_metrics,
        "processed_dir": str(paths["processed_dir"]),
        "run_dir": str(paths["run_dir"]),
        "explanations_dir": str(explanations_dir),
        "clusters_dir": str(clusters_dir),
        "cluster_heatmaps_dir": str(heatmaps_dir),
        "metrics_path": str(paths["metrics"]),
        "predictions_path": str(paths["predictions"]),
        "cluster_assignments_path": str(paths["cluster_assignments"]),
    }


# Edit these two values for an interactive single run.
SELECTED_TARGET = TARGETS[0]
SELECTED_WINDOW = WINDOWS[0]

# Uncomment to run one selected target/window.
# single_result = run_vit_experiment(SELECTED_TARGET, SELECTED_WINDOW, display_heatmaps=True)
# single_result

## Batch Runs

In [ ]:
def run_batch(targets: list[str] = TARGETS, windows: list[str] = WINDOWS, split: str = SPLIT) -> pd.DataFrame:
    results = []
    for window in windows:
        for target in targets:
            logger.info("Batch item started target=%s window=%s", target, window)
            result = run_vit_experiment(target, window, split=split, display_heatmaps=False)
            metrics = flatten_metrics(result.get("evaluation_metrics") or {})
            results.append({
                "target": target,
                "window": window,
                "split": split,
                **metrics,
                "run_dir": result["run_dir"],
                "metrics_path": result["metrics_path"],
                "predictions_path": result["predictions_path"],
                "cluster_assignments_path": result["cluster_assignments_path"],
                "cluster_heatmaps_dir": result["cluster_heatmaps_dir"],
            })
            logger.info("Batch item finished target=%s window=%s", target, window)
    return pd.DataFrame(results)


# Uncomment to run every target/window listed above.
# results_df = run_batch()
# results_df

## Metrics Review

In [ ]:
def collect_existing_metrics(targets: list[str] = TARGETS, windows: list[str] = WINDOWS, split: str = SPLIT) -> pd.DataFrame:
    rows = []
    for window in windows:
        for target in targets:
            paths = experiment_paths(target, window)
            metrics_path = paths["metrics"]
            if not metrics_path.exists():
                logger.info("No metrics found yet for target=%s window=%s at %s", target, window, metrics_path)
                continue
            metrics = flatten_metrics(load_json(metrics_path))
            rows.append({
                "target": target,
                "window": window,
                "split": split,
                **metrics,
                "run_dir": str(paths["run_dir"]),
                "metrics_path": str(metrics_path),
                "predictions_path": str(paths["predictions"]),
                "cluster_assignments_path": str(paths["cluster_assignments"]),
                "cluster_heatmaps_dir": str(paths["cluster_heatmaps"]),
            })
    return pd.DataFrame(rows)


metrics_df = collect_existing_metrics()
metrics_df

## Predictions and Clusters

In [ ]:
REVIEW_TARGET = SELECTED_TARGET
REVIEW_WINDOW = SELECTED_WINDOW

review_paths = experiment_paths(REVIEW_TARGET, REVIEW_WINDOW)

predictions_df = pd.read_csv(review_paths["predictions"]) if review_paths["predictions"].exists() else pd.DataFrame()
cluster_assignments_df = (
    pd.read_csv(review_paths["cluster_assignments"])
    if review_paths["cluster_assignments"].exists()
    else pd.DataFrame()
)

display(predictions_df.head())
display(cluster_assignments_df.head())

## Heatmap Review

In [ ]:
def display_pngs(directory: Path, limit: int | None = None) -> list[Path]:
    paths = sorted(directory.rglob("*.png"))
    if limit is not None:
        paths = paths[:limit]
    if not paths:
        logger.info("No PNG files found under %s", directory)
        return []
    for path in paths:
        print(path.relative_to(directory))
        display(Image(filename=str(path)))
    return paths


display_pngs(review_paths["cluster_heatmaps"], limit=None)

In [ ]:
display_pngs(review_paths["cluster_centroid_heatmaps"], limit=None)